# AI-based variant identification and sequence modelling of Cas proteins
**Google Colab notebook — Python**

Compare existing protein sequences with a chosen reference, identify substitutions and indels, extract ESM-2 embeddings, and explore sequence relationships with PCA. Optional: map observed substitutions to an existing PDB structure.

**Scope:** protein FASTA input, not DNA reads, genome variant calling, Cas-family identification, or experimental activity prediction. Alignment identifies differences; the pretrained AI model represents sequences. No labelled training data are assumed. Embedding distance is not a measure of editing efficiency, safety, stability, or mutation effect. This notebook does not generate new protein designs or predict a new 3D structure.

Use a reference and homologous queries from **one Cas family at a time** (for example, Cas9). Mixing Cas9, Cas12 and Cas13 will make residue correspondence unreliable. Sequence differences and indel positions depend on the alignment, particularly in repeats. Reference positions are 1-based.

### How to run
1. Open this `.ipynb` in https://colab.research.google.com/ using **File → Upload notebook**.
2. Select **Runtime → Change runtime type → T4 GPU** for AI inference. CPU works more slowly.
3. Run the installation cell. Set `DEMO_MODE=False` in the next cell for your data.
4. Upload `reference.fasta` (exactly one amino acid sequence) and `queries.fasta` (one or more sequences; unique IDs).
5. Run cells in order. Download the results ZIP from the final cell.

The default demonstration uses short, artificial sequences. **They are not Cas proteins and support no biological conclusions.**

In [ ]:
%pip -q install "biopython==1.85" "transformers==4.52.2" "numpy>=1.26,<3" "pandas>=2.2,<3" "scikit-learn>=1.4,<2" "matplotlib>=3.8,<4" "py3Dmol>=2.4,<3"

In [ ]:
from pathlib import Path
import io, json, hashlib, platform, shutil, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import SeqIO
from Bio.Align import PairwiseAligner, substitution_matrices
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

DEMO_MODE = True          # Set False to upload your real protein FASTA files.
RUN_AI = True             # Requires internet to download model weights.
MODEL_ID = "facebook/esm2_t6_8M_UR50D"  # Small pretrained ESM-2 baseline.
WINDOW = 900              # Long proteins are covered with overlapping windows.
STRIDE = 450
OUT = Path("cas_results")
OUT.mkdir(exist_ok=True)
# Keep previous runs out of the final ZIP; embedding cache is separate.
for previous in OUT.iterdir():
    if previous.is_file(): previous.unlink()
CACHE = Path("cas_embedding_cache"); CACHE.mkdir(exist_ok=True)

if DEMO_MODE:
    reference_text = ">artificial_reference_NOT_CAS\nMKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQANLQDKPEADKAALEHHHHHH\n"
    ref_demo = str(next(SeqIO.parse(io.StringIO(reference_text), "fasta")).seq)
    queries_text = (
        ">demo_identical\n" + ref_demo + "\n"
        ">demo_substitution\n" + ref_demo[:10] + "A" + ref_demo[11:] + "\n"
        ">demo_insertion\n" + ref_demo[:20] + "GG" + ref_demo[20:] + "\n"
        ">demo_deletion\n" + ref_demo[:30] + ref_demo[32:] + "\n"
    )
else:
    from google.colab import files
    uploaded = files.upload()
    required = {"reference.fasta", "queries.fasta"}
    if not required.issubset(uploaded):
        raise ValueError("Upload both reference.fasta and queries.fasta, with those exact filenames.")
    reference_text = uploaded["reference.fasta"].decode("utf-8-sig")
    queries_text = uploaded["queries.fasta"].decode("utf-8-sig")

AA = set("ACDEFGHIKLMNPQRSTVWY")
def read_proteins(text):
    records = list(SeqIO.parse(io.StringIO(text), "fasta"))
    if not records: raise ValueError("FASTA has no records.")
    result = {}
    for rec in records:
        seq = str(rec.seq).upper()
        if rec.id in result: raise ValueError(f"Duplicate FASTA ID: {rec.id}")
        if not seq or set(seq) - AA:
            raise ValueError(f"{rec.id}: use the 20 standard amino acids; remove gaps/stops and resolve ambiguous residues.")
        if len(seq) < 20: raise ValueError(f"{rec.id}: sequence too short (<20 residues).")
        if set(seq) <= set("ACGT"):
            raise ValueError(f"{rec.id}: looks like DNA, not a protein sequence.")
        result[rec.id] = seq
    return result

reference = read_proteins(reference_text)
queries = read_proteins(queries_text)
if len(reference) != 1: raise ValueError("reference.fasta must contain exactly one record.")
ref_id, ref_seq = next(iter(reference.items()))
if ref_id in queries: raise ValueError("Reference ID must differ from all query IDs.")
sequences = {**reference, **queries}
print(f"Reference: {ref_id}, {len(ref_seq)} aa; queries: {len(queries)}")
if DEMO_MODE: print("DEMONSTRATION ONLY: artificial sequences, not Cas proteins.")

## 1. Identify protein differences
Global alignment uses BLOSUM62 and affine gap penalties. Substitutions use notation such as `Q11A`. Insertions report the reference residue **after which** they occur (`0` means before residue 1); deletions report inclusive reference start/end positions. Identity is identical residue pairs divided by all residue pairs; gaps are reported separately. Low identity is a prompt to inspect homology, not a Cas-classification result.

In [ ]:
aligner = PairwiseAligner()
aligner.mode = "global"
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5
VAR_COLUMNS = ["query_id", "type", "ref_start", "ref_end", "query_start", "query_end", "ref_aa", "query_aa", "notation"]

def compare_proteins(reference_seq, query_seq, query_id):
    alignment = aligner.align(reference_seq, query_seq)[0]
    ri, qi = alignment.indices
    events, paired, identical = [], 0, 0
    j, last_ref = 0, 0
    while j < len(ri):
        r, q = int(ri[j]), int(qi[j])
        if r >= 0 and q >= 0:
            paired += 1; identical += int(reference_seq[r] == query_seq[q])
            last_ref = r + 1
            if reference_seq[r] != query_seq[q]:
                events.append([query_id, "substitution", r+1, r+1, q+1, q+1, reference_seq[r], query_seq[q], f"{reference_seq[r]}{r+1}{query_seq[q]}"])
            j += 1
        else:
            k = j + 1
            kind = "insertion" if r < 0 else "deletion"
            while k < len(ri) and ((ri[k] < 0) if kind == "insertion" else (qi[k] < 0)):
                k += 1
            if kind == "insertion":
                positions = qi[j:k]
                residues = "".join(query_seq[int(p)] for p in positions)
                events.append([query_id, kind, last_ref, last_ref, int(positions[0])+1, int(positions[-1])+1, "", residues, f"after_{last_ref}ins{residues}"])
            else:
                positions = ri[j:k]
                residues = "".join(reference_seq[int(p)] for p in positions)
                start, end = int(positions[0])+1, int(positions[-1])+1
                events.append([query_id, kind, start, end, None, None, residues, "", f"{start}_{end}del"])
                last_ref = end
            j = k
    summary = {"query_id": query_id, "length": len(query_seq), "identity_pct_paired": 100*identical/paired if paired else 0,
               "paired_ref_coverage_pct": 100*paired/len(reference_seq), "alignment_score": float(alignment.score),
               "substitutions": sum(e[1] == "substitution" for e in events),
               "inserted_residues": sum(len(e[7]) for e in events if e[1] == "insertion"),
               "deleted_residues": sum(len(e[6]) for e in events if e[1] == "deletion")}
    return summary, events, alignment

summaries, events = [], []
with (OUT / "alignments.txt").open("w") as handle:
    for qid, qseq in queries.items():
        summary, ev, alignment = compare_proteins(ref_seq, qseq, qid)
        summaries.append(summary); events.extend(ev)
        handle.write(f"\n### {qid}\n{alignment}\n")
summary_df = pd.DataFrame(summaries)
variants_df = pd.DataFrame(events, columns=VAR_COLUMNS)
summary_df.to_csv(OUT / "sequence_summary.csv", index=False)
variants_df.to_csv(OUT / "protein_variants.csv", index=False)
display(summary_df); display(variants_df.head(30))
if (summary_df.identity_pct_paired < 30).any():
    print("Some queries have <30% paired identity. Inspect alignment and family membership before interpreting positions.")

fig, ax = plt.subplots(figsize=(9, 4))
summary_df.set_index("query_id")[["substitutions", "inserted_residues", "deleted_residues"]].plot.bar(ax=ax)
ax.set_ylabel("Observed residue differences"); ax.set_title("Differences relative to reference")
fig.tight_layout(); fig.savefig(OUT / "variant_counts.png", dpi=160); plt.show()

## 2. AI sequence representation
ESM-2 is a pretrained protein language model. The small 8M parameter checkpoint is a practical Colab baseline, not a Cas-specific model.

For long proteins, the code covers the **entire sequence** with overlapping windows, averages representations for residues appearing in multiple windows, then pools over residues. This is an approximation: it loses interactions between residues outside each window and is not equivalent to a full-length model. It does not silently truncate long Cas proteins. Embeddings are cached by sequence, model and window settings. Identical sequences reuse the cache.

In [ ]:
def window_starts(length, window=WINDOW, stride=STRIDE):
    if not 1 <= stride <= window: raise ValueError("Require 1 <= STRIDE <= WINDOW.")
    if length <= window: return [0]
    starts = list(range(0, length-window+1, stride))
    if starts[-1] != length-window: starts.append(length-window)
    return starts

if RUN_AI:
    import torch
    from transformers import AutoTokenizer, AutoModel
    torch.manual_seed(42)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()
    if WINDOW + 2 > model.config.max_position_embeddings:
        raise ValueError("WINDOW exceeds model position capacity.")
    model_commit = getattr(model.config, "_commit_hash", "unknown")
    def embed_protein(seq):
        key = hashlib.sha256(json.dumps([MODEL_ID, model_commit, WINDOW, STRIDE, seq]).encode()).hexdigest()
        cache_file = CACHE / f"{key}.npy"
        if cache_file.exists(): return np.load(cache_file, allow_pickle=False)
        sums = np.zeros((len(seq), model.config.hidden_size), dtype=np.float32)
        counts = np.zeros(len(seq), dtype=np.float32)
        for start in window_starts(len(seq)):
            fragment = seq[start:start+WINDOW]
            tokens = tokenizer(fragment, return_tensors="pt", truncation=False)
            if tokens["input_ids"].shape[1] != len(fragment)+2:
                raise ValueError("Unexpected tokenization: expected one token per amino acid plus two special tokens.")
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.inference_mode():
                hidden = model(**tokens).last_hidden_state[0, 1:len(fragment)+1].float().cpu().numpy()
            sums[start:start+len(fragment)] += hidden
            counts[start:start+len(fragment)] += 1
        if np.any(counts == 0): raise RuntimeError("Window coverage failed.")
        embedding = (sums / counts[:, None]).mean(axis=0)
        np.save(cache_file, embedding)
        return embedding
    embedding_ids = list(sequences)
    embedding_list = []
    for index, (sid, seq) in enumerate(sequences.items(), 1):
        print(f"Embedding {index}/{len(sequences)}: {sid} ({len(seq)} aa)")
        embedding_list.append(embed_protein(seq))
    X = np.stack(embedding_list)
    np.savez_compressed(OUT / "esm_embeddings.npz", ids=np.array(embedding_ids), embeddings=X)
    ref_similarity = cosine_similarity(X, X[:1]).ravel()
    distances_df = pd.DataFrame({"sequence_id": embedding_ids, "cosine_distance_to_reference": np.clip(1-ref_similarity, 0, 2)})
    distances_df.to_csv(OUT / "ai_sequence_distances.csv", index=False)
    display(distances_df)
else:
    print("AI inference disabled; alignment results remain available.")

## 3. Statistical modelling with PCA
PCA shows two directions of greatest embedding variation in this input set. Nearby points have similar pooled embeddings; they are not necessarily similar in function. Axes change when input sequences change. Very small cohorts are illustrative only. No supervised prediction or accuracy claim is possible without relevant experimental labels and leakage-aware validation.

In [ ]:
if RUN_AI and len(X) >= 3 and np.any(np.var(X, axis=0) > 1e-12):
    pca = PCA(n_components=2, random_state=42)
    points = pca.fit_transform(X)
    pca_df = pd.DataFrame(points, columns=["PC1", "PC2"])
    pca_df.insert(0, "sequence_id", embedding_ids)
    pca_df.to_csv(OUT / "pca_coordinates.csv", index=False)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(points[:,0], points[:,1], s=55)
    ax.scatter(points[0,0], points[0,1], marker="*", s=180, label="Reference")
    if len(embedding_ids) <= 30:
        for sid, point in zip(embedding_ids, points): ax.annotate(sid, point, fontsize=8)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
    ax.set_title("ESM-2 sequence embeddings"); ax.legend()
    fig.tight_layout(); fig.savefig(OUT / "esm_pca.png", dpi=160); plt.show()
else:
    print("PCA skipped: enable AI and provide at least 3 sequences with varying embeddings.")

## 4. Optional: map substitutions onto an existing 3D structure
Upload a PDB structure corresponding to the reference. This visualizes an existing experimental or predicted model; it does not perform folding. The code aligns the selected chain's resolved residues to the reference to handle missing residues and numbering offsets. Review the mapping before interpretation. Insertions and deleted/missing residues are not rendered as new atoms.

Set the chain ID and query ID below, then enable this cell. If your goal is full-length structure prediction, use a dedicated folding workflow with suitable compute, then load its PDB here. Do not present a windowed or truncated protein as a full-length structure prediction.

In [ ]:
VIEW_STRUCTURE = False
PDB_CHAIN = "A"
QUERY_TO_SHOW = next(iter(queries))
if VIEW_STRUCTURE:
    from google.colab import files
    from Bio.PDB import PDBParser
    from Bio.SeqUtils import seq1
    import py3Dmol
    pdb_upload = files.upload()
    pdb_names = [name for name in pdb_upload if name.lower().endswith(".pdb")]
    if len(pdb_names) != 1: raise ValueError("Upload exactly one .pdb file.")
    pdb_text = pdb_upload[pdb_names[0]].decode()
    structure = PDBParser(QUIET=True).get_structure("reference", io.StringIO(pdb_text))
    chain = structure[0][PDB_CHAIN]
    residues = [r for r in chain if r.id[0] == " " and "CA" in r]
    structure_seq = "".join(seq1(r.resname, custom_map={"MSE":"M"}, undef_code="X") for r in residues)
    if not structure_seq: raise ValueError("Selected chain has no resolved amino acids.")
    mapping_aligner = PairwiseAligner()
    mapping_aligner.mode = "global"
    mapping_aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
    mapping_aligner.open_gap_score = -10; mapping_aligner.extend_gap_score = -0.5
    mapping = mapping_aligner.align(ref_seq, structure_seq)[0]
    pairs = [(int(r), int(s)) for r,s in zip(*mapping.indices) if r >= 0 and s >= 0]
    identity = sum(ref_seq[r] == structure_seq[s] for r,s in pairs)/max(1,len(pairs))
    print(f"Reference–structure paired identity: {identity:.1%}; inspect the mapping below.")
    print(mapping)
    if identity < .9: raise ValueError("Structure chain differs substantially from reference; select a matching structure.")
    # Map only identical reference residues, avoiding mismatched positions.
    residue_map = {r+1: residues[s] for r,s in pairs if ref_seq[r] == structure_seq[s]}
    rows = []
    for ref_pos, residue in residue_map.items():
        rows.append({"ref_position": ref_pos, "chain": PDB_CHAIN, "pdb_residue": residue.id[1], "insertion_code": residue.id[2]})
    pd.DataFrame(rows).to_csv(OUT / "structure_residue_mapping.csv", index=False)
    if QUERY_TO_SHOW not in queries: raise ValueError("QUERY_TO_SHOW must match a query FASTA ID.")
    changed = variants_df[(variants_df.query_id == QUERY_TO_SHOW) & (variants_df.type == "substitution")]
    viewer = py3Dmol.view(width=850, height=500)
    viewer.addModel(pdb_text, "pdb"); viewer.setStyle({"model":0}, {"cartoon":{"color":"lightgray"}})
    for position in changed.ref_start:
        residue = residue_map.get(int(position))
        if residue is not None:
            selector = {"chain":PDB_CHAIN,"resi":residue.id[1],"icode":residue.id[2].strip()}
            viewer.addStyle(selector, {"stick":{"color":"red"}})
    viewer.zoomTo(); viewer.show()

## 5. Save and download results
The ZIP contains alignment differences, sequence summaries, charts and, if enabled, AI embeddings/PCA and structure mapping. It also includes run settings, package versions and sequence hashes. Back up downloaded results because Colab runtimes are temporary.

For a labelled activity model, you would additionally need measured outcomes and sample provenance, and should split related sequence groups together to avoid inflated performance from near-identical sequences in training and testing.

In [ ]:
manifest = {
    "demo_mode": DEMO_MODE, "run_ai": RUN_AI, "reference_id": ref_id,
    "model_id": MODEL_ID if RUN_AI else None,
    "model_commit": model_commit if RUN_AI else None,
    "window": WINDOW, "stride": STRIDE,
    "alignment": {"mode":"global", "matrix":"BLOSUM62", "gap_open":-10, "gap_extend":-.5},
    "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name) for name in ["biopython", "numpy", "pandas", "scikit-learn", "transformers", "torch"]},
    "sequences": {sid: {"length":len(seq), "sha256":hashlib.sha256(seq.encode()).hexdigest()} for sid,seq in sequences.items()},
    "interpretation": "Exploratory sequence analysis. No functional or clinical claims. Long-sequence embeddings use window approximation."
}
(OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2))
archive = shutil.make_archive("cas_analysis_results", "zip", OUT)
print("Saved:", archive)
from google.colab import files
files.download(archive)

## Sources and practical limits
- Biopython pairwise alignment documentation: https://biopython.org/docs/1.85/Tutorial/chapter_pairwise.html
- Hugging Face ESM API documentation: https://huggingface.co/docs/transformers/v4.52.2/model_doc/esm
- ESM project and pretrained-model background: https://github.com/facebookresearch/esm
- Model checkpoint: https://huggingface.co/facebook/esm2_t6_8M_UR50D

This is a starting workflow for uploaded sequences. Alignment is quadratic in sequence length and each query is processed serially; it is not a distributed pipeline for millions of proteins. The first AI run downloads pretrained weights. GPU availability and Colab memory vary. Full-length Cas proteins are supported by the windowed representation, with the limitations explained above.

**Validation of this deliverable:** notebook schema, Python cell syntax, and reference-coordinate variant handling are checked locally with artificial substitutions and terminal/internal indels. Real Colab upload/download interactions and pretrained-model inference require running in Colab; they have not been claimed as locally verified.